In [ ]:
%matplotlib inline
import pandas as pd
import scipy
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import sktime
import sklearn
from scipy.io import loadmat
from sktime.transformations.panel.rocket import Rocket
import timesynth as ts
import random
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

In [ ]:
n_steps = 3600
# get original segment for given sample that is impacted by a kernel
def get_original_segment(sample, impacted_k_weights,
    impacted_k_bias,
    impacted_k_dilation,
    impacted_k_padding,
    impacted_k_lengths):
    sample_data = X_test[sample, 0, :]

    # check for padding and apply to the segment if needed on both left and right sides
    if impacted_k_padding != 0:
        padding_left = ((impacted_k_lengths - 1) * impacted_k_dilation) // 2
        padding_right = ((impacted_k_lengths - 1) * impacted_k_dilation) - padding_left
        padded_segment = np.pad(sample_data, (padding_left, padding_right), mode='constant')
    else:
        padded_segment = sample_data.copy()
        
    conv = np.zeros(n_steps)
    for i in range(n_steps):
        indices = i + np.arange(impacted_k_lengths) * impacted_k_dilation
        if np.all(indices < len(padded_segment)):
            conv[i] = np.sum(padded_segment[indices] * impacted_k_weights)
    return conv + impacted_k_bias

# get injected segment
def get_injected_segment(sample, impacted_k_weights,
    impacted_k_bias,
    impacted_k_dilation,
    impacted_k_padding,
    impacted_k_lengths):
    sample_data = X_train[sample, 0, :]

    # check for padding and apply to the segment if needed on both left and right sides
    if impacted_k_padding != 0:
        padding_left = ((impacted_k_lengths - 1) * impacted_k_dilation) // 2
        padding_right = ((impacted_k_lengths - 1) * impacted_k_dilation) - padding_left
        padded_segment = np.pad(sample_data, (padding_left, padding_right), mode='constant')
    else:
        padded_segment = sample_data.copy()

    conv = np.zeros(n_steps)
    for i in range(n_steps):
        indices = i + np.arange(impacted_k_lengths) * impacted_k_dilation
        if np.all(indices < len(padded_segment)):
            conv[i] = np.sum(padded_segment[indices] * impacted_k_weights)
    return conv + impacted_k_bias


In [ ]:
final_df = pd.read_csv('C:/Users/satavans/Downloads/non_aug_data_all60fps.csv')

In [ ]:
# alcohol drive
final_lpupila = pd.DataFrame(final_df.loc[
    (final_df['Subject'].astype(str) != '5') &
    (final_df['Subject'].astype(str) != '7') &
    (final_df['Subject'].astype(str) != '9') &
    (final_df['Subject'].astype(str) != '10') &
    (final_df['Subject'].astype(str) != '11') &
    (final_df['Subject'].astype(str) != '15') &
    (final_df['Subject'].astype(str) != '17') &
    (final_df['Subject'].astype(str) != '24') &
    (final_df['Subject'].astype(str) != '25') &
    (final_df['Subject'].astype(str) != '35') &
    (final_df['Drive'].astype(str) == '2'),
    ['LPupil_Diameter', 'Subject']
])

# alcohol drive
final_lpupils = pd.DataFrame(final_df.loc[
    (final_df['Subject'].astype(str) != '5') &
    (final_df['Subject'].astype(str) != '7') &
    (final_df['Subject'].astype(str) != '9') &
    (final_df['Subject'].astype(str) != '10') &
    (final_df['Subject'].astype(str) != '11') &
    (final_df['Subject'].astype(str) != '15') &
    (final_df['Subject'].astype(str) != '17') &
    (final_df['Subject'].astype(str) != '24') &
    (final_df['Subject'].astype(str) != '25') &
    (final_df['Subject'].astype(str) != '35') &
    (final_df['Drive'].astype(str) == '1'),
    ['LPupil_Diameter', 'Subject']
])



In [ ]:
# 1. Define dimensions
n_steps = 3600
n_samples = len(final_lpupila) // n_steps
n_samples2 = len(final_lpupils) // n_steps
n_vars = 1

In [ ]:
n_samples = len(final_lpupila) // n_steps

data_3d2 = final_lpupila['LPupil_Diameter'].values.reshape(
    n_samples, 1, n_steps
)

subject_ids_alcohol = (
    final_lpupila['Subject']
    .values
    .reshape(n_samples, n_steps)[:,0]
)

In [ ]:
n_samples2 = len(final_lpupils) // n_steps

data_3d = final_lpupils['LPupil_Diameter'].values.reshape(
    n_samples2, 1, n_steps
)

subject_ids_sober = (
    final_lpupils['Subject']
    .values
    .reshape(n_samples2, n_steps)[:,0]
)

In [ ]:
y = [0] * (data_3d.shape[0]) + [1] * (data_3d2.shape[0])

In [ ]:
subject_ids = np.concatenate([
    subject_ids_sober,
    subject_ids_alcohol
])

combined_data_3d = np.concatenate([data_3d, data_3d2], axis=0)


In [ ]:
original_auc = []
original_accuracy = []
# generating the splits and finding original aucs
for i in range(100):
    splitter = GroupShuffleSplit(
    test_size=0.2,
    random_state= i
    )
    splitter.split(combined_data_3d, y, groups=subject_ids)
    train_idx, test_idx = next(splitter.split(combined_data_3d, y, groups=subject_ids))
    X_train = combined_data_3d[train_idx]
    X_test = combined_data_3d[test_idx]

    y_train = np.array(y)[train_idx]
    y_test = np.array(y)[test_idx]

    # fit the rocket transformer on sober drive data
    trf = Rocket(num_kernels=1000, random_state = 1234)
    rocket_features = trf.fit_transform(X_train)
    model = LogisticRegression(max_iter = 1000)
    scaler = StandardScaler()
    rocket_features = scaler.fit_transform(rocket_features)
    model.fit(rocket_features, y_train)
    X_test_rocket = trf.transform(X_test)
    X_test_rocket = scaler.transform(X_test_rocket)

    y_pred = model.predict(X_test_rocket)
    original_accuracy.append(accuracy_score(y_pred, y_test))
    print("Original:", accuracy_score(y_pred, y_test))
    y_prob = model.predict_proba(X_test_rocket)[:,1]
    original_auc.append(roc_auc_score(y_test, y_prob))
    print("Original:", roc_auc_score(y_test, y_prob))
    
    pd.DataFrame(rocket_features).to_csv('C:/Users/satavans/Downloads/alcohol_application/train_files/driving_split' + str(i) + '.csv', index=False)
    pd.DataFrame(y_train).to_csv('C:/Users/satavans/Downloads/alcohol_application/train_files/driving_split_y' + str(i) + '.csv', index=False)
    np.save("C:/Users/satavans/Downloads/alcohol_application/train_raw/X_train" + str(i) + ".npy", X_train)
    np.save("C:/Users/satavans/Downloads/alcohol_application/test_raw/X_test" + str(i) + ".npy", X_test)
    np.save("C:/Users/satavans/Downloads/alcohol_application/train_raw/y_train" + str(i) + ".npy", y_train)
    np.save("C:/Users/satavans/Downloads/alcohol_application/test_raw/y_test" + str(i) + ".npy", y_test)


In [ ]:
auc_75 = []
accuracy_75 = []
auc_50 = []
accuracy_50 = []
auc_25 = []
accuracy_25 = []
# deletion test for FDF-ROCKET
for i in range(10):
    rocket_features = pd.read_csv('C:/Users/satavans/Downloads/alcohol_application/train_files/driving_split' + str(i) + '.csv')
    X_train = np.load("C:/Users/satavans/Downloads/alcohol_application/train_raw/X_train" + str(i) + ".npy")
    X_test = np.load("C:/Users/satavans/Downloads/alcohol_application/test_raw/X_test" + str(i) + ".npy")
    y_train = np.load("C:/Users/satavans/Downloads/alcohol_application/train_raw/y_train" + str(i) + ".npy")
    y_test = np.load("C:/Users/satavans/Downloads/alcohol_application/test_raw/y_test" + str(i) + ".npy")
    X_test2 = X_test.copy()

    # fit the rocket transformer on sober drive data
    trf = Rocket(num_kernels=1000, random_state = 1234)
    rocket_features = trf.fit_transform(X_train)
    model = LogisticRegression(max_iter = 1000)
    scaler = StandardScaler()
    rocket_features = scaler.fit_transform(rocket_features)
    model.fit(rocket_features, y_train)

    X_test_rocket = trf.transform(X_test)
    X_test_rocket = scaler.transform(X_test_rocket)
    y_pred = model.predict(X_test_rocket)
    print("Original Accuracy:", accuracy_score(y_pred, y_test))
    y_prob = model.predict_proba(X_test_rocket)[:,1]

    print("Original AUC: ", roc_auc_score(y_test, y_prob))
    # getting the kernel params
    kernel_weights = trf.kernels[0]
    kernel_lengths = trf.kernels[1]
    kernel_biases = trf.kernels[2]
    kernel_dilations = trf.kernels[3]
    kernel_paddings = trf.kernels[4]
    kernels = pd.read_csv("C:/Users/satavans/Downloads/alcohol_application/kernels/spike_kernels" + str(i) + ".csv")
    selected_kernels = kernels['selected_features']

    total_time_points = X_train.shape[2]

    all_points_flagged = set()
    for sample in range(X_test.shape[0]):
        all_point_diffs = {}
        any_kernel_flagged = set()
        for kernel in selected_kernels:
            kernel_start = np.sum(kernel_lengths[:kernel])
            kernel_end = kernel_start + kernel_lengths[kernel]

            impacted_k_weights = kernel_weights[kernel_start:kernel_end]
            impacted_k_bias = kernel_biases[kernel]
            impacted_k_dilation = kernel_dilations[kernel]
            impacted_k_padding = kernel_paddings[kernel]
            impacted_k_lengths = kernel_lengths[kernel]

            baseline = get_original_segment(sample, impacted_k_weights,
                                            impacted_k_bias,
                                            impacted_k_dilation,
                                            impacted_k_padding,
                                            impacted_k_lengths)
            # store maximum diff seen for each timepoint
            for idx, value in enumerate(baseline):
                if idx not in all_point_diffs:
                    all_point_diffs[idx] = value
                else:
                    all_point_diffs[idx] = all_point_diffs.get(idx, 0) + value

        # get top k (# of spikes) points
        n_points = len(all_point_diffs)
        n_del = int(0.25 * n_points)
        # len(changed_set)
        any_kernel_flagged = set(
            sorted(all_point_diffs, key=all_point_diffs.get, reverse=True)[:n_del]
        )
        mu = np.mean(X_test[sample])
        sigma = np.std(X_test[sample])

        X_test2[sample,0,list(any_kernel_flagged)] = mu

    X_test_rocket2 = trf.transform(X_test2)
    X_test_rocket2 = scaler.transform(X_test_rocket2)
    y_pred2 = model.predict(X_test_rocket2)
    print("Top 75 Accuracy:", accuracy_score(y_pred2, y_test))
    accuracy_75.append(accuracy_score(y_pred2, y_test))
    y_prob2 = model.predict_proba(X_test_rocket2)[:,1]

    print("Top 25 removed AUC: ", roc_auc_score(y_test, y_prob2))
    auc_75.append(roc_auc_score(y_test, y_prob2))

    X_test2 = X_test.copy()
    all_points_flagged = set()
    for sample in range(X_test.shape[0]):
        all_point_diffs = {}
        any_kernel_flagged = set()
        for kernel in selected_kernels:
            kernel_start = np.sum(kernel_lengths[:kernel])
            kernel_end = kernel_start + kernel_lengths[kernel]

            impacted_k_weights = kernel_weights[kernel_start:kernel_end]
            impacted_k_bias = kernel_biases[kernel]
            impacted_k_dilation = kernel_dilations[kernel]
            impacted_k_padding = kernel_paddings[kernel]
            impacted_k_lengths = kernel_lengths[kernel]

            baseline = get_original_segment(sample, impacted_k_weights,
                                            impacted_k_bias,
                                            impacted_k_dilation,
                                            impacted_k_padding,
                                            impacted_k_lengths)
            # store maximum diff seen for each timepoint
            for idx, value in enumerate(baseline):
                if idx not in all_point_diffs:
                    all_point_diffs[idx] = value
                else:
                    all_point_diffs[idx] = all_point_diffs.get(idx, 0) + value

        # get top k (# of spikes) points
        n_points = len(all_point_diffs)
        n_del = int(0.5 * n_points)
        # len(changed_set)
        any_kernel_flagged = set(
            sorted(all_point_diffs, key=all_point_diffs.get, reverse=True)[:n_del]
        )
        mu = np.mean(X_test[sample])
        sigma = np.std(X_test[sample])

        X_test2[sample,0,list(any_kernel_flagged)] = mu

    X_test_rocket2 = trf.transform(X_test2)
    X_test_rocket2 = scaler.transform(X_test_rocket2)
    y_pred2 = model.predict(X_test_rocket2)
    print("Top 50 removed Accuracy:", accuracy_score(y_pred2, y_test))
    accuracy_50.append(accuracy_score(y_pred2, y_test))
    y_prob2 = model.predict_proba(X_test_rocket2)[:,1]

    print("Top 50 removed AUC: ", roc_auc_score(y_test, y_prob2))
    auc_50.append(roc_auc_score(y_test, y_prob2))
    
    X_test2 = X_test.copy()
    all_points_flagged = set()
    for sample in range(X_test.shape[0]):
        all_point_diffs = {}
        any_kernel_flagged = set()
        for kernel in selected_kernels:
            kernel_start = np.sum(kernel_lengths[:kernel])
            kernel_end = kernel_start + kernel_lengths[kernel]

            impacted_k_weights = kernel_weights[kernel_start:kernel_end]
            impacted_k_bias = kernel_biases[kernel]
            impacted_k_dilation = kernel_dilations[kernel]
            impacted_k_padding = kernel_paddings[kernel]
            impacted_k_lengths = kernel_lengths[kernel]

            baseline = get_original_segment(sample, impacted_k_weights,
                                            impacted_k_bias,
                                            impacted_k_dilation,
                                            impacted_k_padding,
                                            impacted_k_lengths)
            # store maximum diff seen for each timepoint
            for idx, value in enumerate(baseline):
                if idx not in all_point_diffs:
                    all_point_diffs[idx] = value
                else:
                    all_point_diffs[idx] = all_point_diffs.get(idx, 0) + value

        # get top k (# of spikes) points
        n_points = len(all_point_diffs)
        n_del = int(0.75 * n_points)
        # len(changed_set)
        any_kernel_flagged = set(
            sorted(all_point_diffs, key=all_point_diffs.get, reverse=True)[:n_del]
        )
        mu = np.mean(X_test[sample])
        sigma = np.std(X_test[sample])

        X_test2[sample,0,list(any_kernel_flagged)] = mu

    X_test_rocket2 = trf.transform(X_test2)
    X_test_rocket2 = scaler.transform(X_test_rocket2)
    y_pred2 = model.predict(X_test_rocket2)
    print("Top 75 removed Accuracy:", accuracy_score(y_pred2, y_test))
    accuracy_25.append(accuracy_score(y_pred2, y_test))
    y_prob2 = model.predict_proba(X_test_rocket2)[:,1]

    print("Top 75 removed AUC: ", roc_auc_score(y_test, y_prob2))
    auc_25.append(roc_auc_score(y_test, y_prob2))

In [ ]:
pd.DataFrame(auc_75).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_results/auc_75.csv', index=False)
pd.DataFrame(auc_25).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_results/auc_25.csv', index=False)
pd.DataFrame(auc_50).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_results/auc_50.csv', index=False)

pd.DataFrame(accuracy_25).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_results/accuracy_25.csv', index=False)
pd.DataFrame(accuracy_50).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_results/accuracy_50.csv', index=False)
pd.DataFrame(accuracy_75).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_results/accuracy_75.csv', index=False)

In [ ]:
print("AUC = ", np.mean(auc_75), ",",  np.mean(auc_50), ",",  np.mean(auc_25))
print("Accuracy = ", np.mean(accuracy_75), ",",  np.mean(accuracy_50), ",",  np.mean(accuracy_25))

In [ ]:
# random
print("AUC = ", np.mean(auc_75), ",",  np.mean(auc_50), ",",  np.mean(auc_25))
print("Accuracy = ", np.mean(accuracy_75), ",",  np.mean(accuracy_50), ",",  np.mean(accuracy_25))

In [ ]:
# random
print("AUC = ", np.mean(auc_25), ",",  np.mean(auc_50), ",",  np.mean(auc_75))
print("Accuracy = ", np.mean(accuracy_25), ",",  np.mean(accuracy_50), ",",  np.mean(accuracy_75))

In [ ]:
np.mean(accuracy_25)

In [ ]:
beta = model.coef_[0]
beta

In [ ]:
# 271m 45.1s
auc_75 = []
accuracy_75 = []
auc_50 = []
accuracy_50 = []
auc_25 = []
accuracy_25 = []
# deletion test for ROCKET-LRP
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, r"C:\Users\satavans\Downloads\rocket-lrp")
from rocket_lrp.rocket import RocketTransform
from rocket_lrp.lrp import RocketLRPExplainer
from rocket_lrp.explainers import LinearSHAPExplainer


def filter_latent_importances(R_h, alpha_percent=30.0):
    """Zero the lowest alpha_percent of absolute latent importances per sample."""
    R_h = np.asarray(R_h, dtype=np.float64)
    if alpha_percent <= 0:
        return R_h
    if R_h.ndim == 1:
        R_h = R_h[None, :]

    out = R_h.copy()
    mag = np.abs(out)
    thr = np.percentile(mag, alpha_percent, axis=1, keepdims=True)
    out[mag < thr] = 0.0
    return out


def explain_with_paper_lrp(train_X3, test_X, test_X2, y, alpha_percent=30.0, epsilon=1e-6):
    """Run the paper's ROCKET + LinearSHAP + LRP workflow exactly for one split."""

    trf = Rocket(num_kernels=1000, random_state=1234)
    trf.fit(train_X3)

    rocket = RocketTransform.from_sktime(
        trf,
        train_X3
    )

    H_train = trf.transform(train_X3)
    scaler = StandardScaler().fit(H_train)
    H_train_scaled = scaler.transform(H_train)

    classifier = LogisticRegression(
        max_iter=1000,
    ).fit(H_train_scaled, y)

    # Stage 1: paper-style LinearSHAP in the latent ROCKET feature space.
    baseline = H_train_scaled.mean(axis=0)
    stage1 = LinearSHAPExplainer(
        classifier.coef_.ravel(),
        baseline=baseline,
        scaler_mean=scaler.mean_,
        scaler_scale=scaler.scale_,
    )

    H_test = trf.transform(test_X2)
    H_test_scaled = scaler.transform(H_test)
    R_h = stage1.explain(H_test_scaled)

    # Stage 2: paper-style LRP propagation to the original input space.
    lrp = RocketLRPExplainer(rocket, epsilon=epsilon, alpha=alpha_percent)
    R_h = filter_latent_importances(R_h, alpha_percent=alpha_percent)
    R_lrp = lrp.explain(test_X2, R_h)

    return classifier, scaler, trf, R_lrp


precision_all = []
recall_all = []
fdr_all = []
f1_all = []

for i in range(10):
    rocket_features = pd.read_csv('C:/Users/satavans/Downloads/alcohol_application/train_files/driving_split' + str(i) + '.csv')
    X_train = np.load("C:/Users/satavans/Downloads/alcohol_application/train_raw/X_train" + str(i) + ".npy")
    X_test = np.load("C:/Users/satavans/Downloads/alcohol_application/test_raw/X_test" + str(i) + ".npy")
    y_train = np.load("C:/Users/satavans/Downloads/alcohol_application/train_raw/y_train" + str(i) + ".npy")
    y_test = np.load("C:/Users/satavans/Downloads/alcohol_application/test_raw/y_test" + str(i) + ".npy")
    X_train2 = X_train.copy()

    classifier, scaler, rocket, R_lrp = explain_with_paper_lrp(
        X_train,
        X_test,
        X_test,
        y_train,
        alpha_percent=30.0,
        epsilon=1e-6,
    )


    # Transform original test set once
    H_test = rocket.transform(X_test)
    H_test_scaled = scaler.transform(H_test)

    y_prob_original = classifier.predict_proba(H_test_scaled)[:,1]
    print("Original AUC:", roc_auc_score(y_test, y_prob_original))


    # ---------- Remove 25% ----------
    X_test2 = X_test.copy()

    for sample in range(X_test.shape[0]):
        rel = np.abs(R_lrp[sample,0,:])

        n_remove = int(len(rel) * 0.25)

        remove_idx = np.argsort(rel)[-n_remove:]

        mu = np.mean(X_test[sample])
        sigma = np.std(X_test[sample])

        X_test2[sample,0,remove_idx] = mu

    H_test2 = rocket.transform(X_test2)
    H_test2_scaled = scaler.transform(H_test2)

    y_prob = classifier.predict_proba(H_test2_scaled)[:,1]

    print("Remove 25% AUC:",
        roc_auc_score(y_test, y_prob))

    auc_25.append(
        roc_auc_score(y_test,y_prob)
    )
    pred = classifier.predict(H_test2_scaled)

    print("Accuracy:", accuracy_score(y_test, pred))
    accuracy_25.append(accuracy_score(y_test, pred))


    # ---------- Remove 50% ----------
    X_test2 = X_test.copy()

    for sample in range(X_test.shape[0]):
        rel = np.abs(R_lrp[sample,0,:])

        n_remove = int(len(rel) * 0.50)

        remove_idx = np.argsort(rel)[-n_remove:]

        mu = np.mean(X_test[sample])
        sigma = np.std(X_test[sample])

        X_test2[sample,0,remove_idx] = mu


    H_test2 = rocket.transform(X_test2)
    H_test2_scaled = scaler.transform(H_test2)

    y_prob = classifier.predict_proba(H_test2_scaled)[:,1]

    print("Remove 50% AUC:",
        roc_auc_score(y_test,y_prob))

    auc_50.append(
        roc_auc_score(y_test,y_prob)
    )
     
    pred = classifier.predict(H_test2_scaled)

    print("Accuracy:", accuracy_score(y_test, pred))
    accuracy_50.append(accuracy_score(y_test, pred))



    # ---------- Remove 75% ----------
    X_test2 = X_test.copy()

    for sample in range(X_test.shape[0]):
        rel = np.abs(R_lrp[sample,0,:])

        n_remove = int(len(rel) * 0.75)

        remove_idx = np.argsort(rel)[-n_remove:]

        mu = np.mean(X_test[sample])
        sigma = np.std(X_test[sample])

        X_test2[sample,0,remove_idx] = mu


    H_test2 = rocket.transform(X_test2)
    H_test2_scaled = scaler.transform(H_test2)

    y_prob = classifier.predict_proba(H_test2_scaled)[:,1]

    print("Remove 75% AUC:",
        roc_auc_score(y_test,y_prob))

    auc_75.append(
        roc_auc_score(y_test,y_prob)
    )
    pred = classifier.predict(H_test2_scaled)

    print("Accuracy:", accuracy_score(y_test, pred))
    accuracy_75.append(accuracy_score(y_test, pred))

In [ ]:
pd.DataFrame(auc_75).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_lrp_results/auc_75.csv', index=False)
pd.DataFrame(auc_25).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_lrp_results/auc_25.csv', index=False)
pd.DataFrame(auc_50).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_lrp_results/auc_50.csv', index=False)

pd.DataFrame(accuracy_25).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_lrp_results/accuracy_25.csv', index=False)
pd.DataFrame(accuracy_50).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_lrp_results/accuracy_50.csv', index=False)
pd.DataFrame(accuracy_75).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_lrp_results/accuracy_75.csv', index=False)